# Optimal Transport based ICA versus FastICA - linear setting

### We compare the performance of OT based ICA and FastICA over varying number of dimensions and sample size of simulation over a grid With Gradient search and symmetric orthogonalization, i.e. find all IC candidates at once by enforcing global orthogonalization.

In [1]:
import numpy as np
import torch
import pandas as pd
import plotly.graph_objects as go
from sklearn.decomposition import FastICA
from tqdm.notebook import tqdm

# Import your updated class
from wasserstein_ica import WassersteinICA

In [2]:
# ==========================================
# 1. Configuration
# ==========================================
# Dimensions: 2 to 10
DIMENSION_RANGE = list(range(2, 11))

# Samples: 100, then 1000 to 10000 in steps of 1000
SAMPLE_SIZE_RANGE = [100] + list(range(1000, 10001, 1000))

# Trials per grid point
N_TRIALS = 5

print(f"--- Grid Search Configuration ---")
print(f"Dimensions: {DIMENSION_RANGE}")
print(f"Samples: {SAMPLE_SIZE_RANGE}")
print(f"Total Runs: {len(DIMENSION_RANGE) * len(SAMPLE_SIZE_RANGE) * N_TRIALS * 2}")

--- Grid Search Configuration ---
Dimensions: [2, 3, 4, 5, 6, 7, 8, 9, 10]
Samples: [100, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]
Total Runs: 990


In [3]:
# ==========================================
# 1. Helper Functions (Redefine ALL of them)
# ==========================================

def get_whitening_matrix(X_torch):
    """
    Calculates the Whitening Matrix externally.
    Use this to avoid the 'AttributeError: W_white' issue.
    """
    n_samples = X_torch.shape[1]
    X_centered = X_torch - torch.mean(X_torch, dim=1, keepdim=True)
    cov = torch.matmul(X_centered, X_centered.t()) / (n_samples - 1)
    D, E = torch.linalg.eigh(cov)
    # Add epsilon for numerical stability
    D_inv_sqrt = torch.diag(1.0 / torch.sqrt(D + 1e-5))
    # Return W (Whitening Matrix)
    return torch.matmul(D_inv_sqrt, E.T).cpu().numpy()

def amari_error(W_est, A_true):
    """
    Computes the Amari Index (Lower is Better).
    """
    if W_est is None or np.any(np.isnan(W_est)): return np.nan
    P = np.abs(W_est @ A_true)
    n = P.shape[0]
    row_sum = np.sum(P, axis=1)
    row_max = np.max(P, axis=1)
    term1 = np.sum((row_sum / row_max) - 1)
    col_sum = np.sum(P, axis=0)
    col_max = np.max(P, axis=0)
    term2 = np.sum((col_sum / col_max) - 1)
    return (term1 + term2) / (2 * n * (n - 1))

def generate_dataset(n_dim, n_samples, seed=None):
    """
    Generates mixed data (X) and returns the mixing matrix (A).
    MUST return (X, A) to avoid unpacking errors.
    """
    if seed is not None:
        np.random.seed(seed)
        torch.manual_seed(seed)
    
    sources = []
    # Cycle through 4 distribution types
    for i in range(n_dim):
        dist_type = i % 4
        if dist_type == 0: s = np.random.laplace(0, 1, n_samples)
        elif dist_type == 1: s = np.random.uniform(-1.73, 1.73, n_samples)
        elif dist_type == 2: s = np.random.standard_t(df=3, size=n_samples)
        else: 
            s = np.random.beta(0.5, 0.5, size=n_samples)
            s = (s - np.mean(s)) / np.std(s)
        sources.append(s)
        
    S = np.stack(sources)
    
    # Random Mixing Matrix
    cond_num = 1000
    while cond_num > 100:
        A = np.random.randn(n_dim, n_dim)
        cond_num = np.linalg.cond(A)
        
    X = A @ S
    
    # CRITICAL: Returns a tuple of 2 items
    return torch.tensor(X, dtype=torch.float32), A

In [4]:
# ==========================================
# 3. Main Execution Loop
# ==========================================
results = []
pbar = tqdm(total=len(DIMENSION_RANGE) * len(SAMPLE_SIZE_RANGE) * N_TRIALS)

for dim in DIMENSION_RANGE:
    for n in SAMPLE_SIZE_RANGE:
        for trial in range(N_TRIALS):
            seed = trial + (dim * 100) + (n * 10)
            
            # Generate Data
            X_torch, A_true = generate_dataset(n_dim=dim, n_samples=n, seed=seed)
            X_np = X_torch.numpy()
            
            # --- Method 1: FastICA ---
            try:
                fast_ica = FastICA(n_components=dim, algorithm='parallel', 
                                   max_iter=2000, tol=1e-3, random_state=seed)
                fast_ica.fit(X_np.T)
                W_fast = fast_ica.components_ 
                score_fast = amari_error(W_fast, A_true)
            except Exception as e:
                print(f"Error: {e}")
                score_fast = np.nan
            
            # --- Method 2: WassersteinICA ---
            try:
                ica = WassersteinICA(X_torch)
                ica.whiten()
                
                # Optimize Symmetric
                # Note: If this fails with "no attribute optimize_symmetric",
                # you MUST restart your kernel.
                W_sphere = ica.optimize_symmetric(
                    n_components=dim, 
                    max_iter=300, 
                    lr=0.1
                )
                
                # --- FIX: Calculate W_white externally ---
                W_white = get_whitening_matrix(X_torch)
                
                # Reconstruct Total
                W_wass = W_sphere.cpu().numpy() @ W_white
                
                score_wass = amari_error(W_wass, A_true)
            except Exception as e:
                print(f"Error: {e}") 
                score_wass = np.nan
            
            # Store
            results.append({'Dimension': dim, 'Samples': n, 'Method': 'FastICA', 'Amari': score_fast})
            results.append({'Dimension': dim, 'Samples': n, 'Method': 'WassersteinICA (Sym)', 'Amari': score_wass})
            
            pbar.update(1)

pbar.close()
df_results = pd.DataFrame(results)

  0%|          | 0/495 [00:00<?, ?it/s]

/home/ashutoshjha/py_envs/ot_ica/lib/python3.12/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
/home/ashutoshjha/py_envs/ot_ica/lib/python3.12/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
/home/ashutoshjha/py_envs/ot_ica/lib/python3.12/site-packages/sklearn/decomposition/_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


In [5]:
# ==========================================
# 4. 3D Surface Plot
# ==========================================
df_avg = df_results.groupby(['Dimension', 'Samples', 'Method'])['Amari'].mean().reset_index()

pivot_fast = df_avg[df_avg['Method'] == 'FastICA'].pivot(index='Dimension', columns='Samples', values='Amari')
pivot_wass = df_avg[df_avg['Method'] == 'WassersteinICA (Sym)'].pivot(index='Dimension', columns='Samples', values='Amari')

x_vals = pivot_fast.columns
y_vals = pivot_fast.index

fig = go.Figure()

# FastICA (Blue)
fig.add_trace(go.Surface(
    z=pivot_fast.values, x=x_vals, y=y_vals,
    colorscale='Blues', opacity=0.8, name='FastICA',
    showscale=False
))

# Wasserstein Symmetric (Red)
fig.add_trace(go.Surface(
    z=pivot_wass.values, x=x_vals, y=y_vals,
    colorscale='Reds', opacity=0.8, name='WassersteinICA',
    showscale=True, colorbar=dict(title='Amari Error')
))

fig.update_layout(
    title='Symmetric ICA Comparison: Wasserstein (Red) vs FastICA (Blue)',
    scene=dict(
        xaxis_title='Sample Size (Log Scale)',
        yaxis_title='Dimensions',
        zaxis_title='Amari Error',
        xaxis_type='log',
        zaxis=dict(range=[0, 0.6])
    ),
    width=900, height=700,
)

fig.show()